# T2.L2. Математична модель транспортної задачі

Мета: пройти шлях **дані → математична модель → PuLP → verification → scenarios → interpretation**.


## 1. Постановка

Три джерела `S1–S3` мають сумарно 125 одиниць ресурсу. Чотири пункти `D1–D4` потребують сумарно 125 одиниць. Потрібно мінімізувати сумарні транспортні витрати.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

LESSON = Path.cwd()
if LESSON.name == "notebooks":
    LESSON = LESSON.parent
SRC = LESSON / "src"
sys.path.insert(0, str(SRC))

from model import (
    validate_problem, solve_transport_pulp, solve_transport_scipy,
    check_solution, total_cost, long_plan
)


## 2. Завантаження даних


In [ ]:
costs = pd.read_csv(LESSON / "data" / "costs.csv", index_col="supplier").astype(float)
supply = pd.read_csv(LESSON / "data" / "supply.csv").set_index("supplier")["supply"].astype(float)
demand = pd.read_csv(LESSON / "data" / "demand.csv").set_index("consumer")["demand"].astype(float)

display(costs)
display(supply.to_frame())
display(demand.to_frame())


In [ ]:
print("Total supply:", supply.sum())
print("Total demand:", demand.sum())
validate_problem(costs, supply, demand)


## 3. Математична модель

\[
Z=\sum_i\sum_j c_{ij}x_{ij}\to\min
\]

за балансів джерел і пунктів потреби та \(x_{ij}\ge0\).


## 4. Розв'язання через PuLP


In [ ]:
result = solve_transport_pulp(costs, supply, demand)
print("Status:", result.status)
print("Total cost:", result.total_cost)
display(result.plan)


Контрольне значення baseline: **515**.


## 5. Незалежна перевірка допустимості


In [ ]:
verification = check_solution(result.plan, supply, demand)
verification


In [ ]:
print("Recomputed cost:", total_cost(result.plan, costs))
display(long_plan(result.plan, costs))


## 6. Незалежний solver


In [ ]:
scipy_result = solve_transport_scipy(costs, supply, demand)
print("PuLP:", result.total_cost)
print("SciPy:", scipy_result.total_cost)
print("Difference:", abs(result.total_cost - scipy_result.total_cost))


Збіг двох solver підсилює довіру до **обчислення**, але не доводить адекватність вихідних припущень реальному об'єкту.


## 7. Візуалізація baseline


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(result.plan.to_numpy(), aspect="auto")
ax.set_xticks(range(len(result.plan.columns)), result.plan.columns)
ax.set_yticks(range(len(result.plan.index)), result.plan.index)
ax.set_xlabel("Пункт потреби")
ax.set_ylabel("Джерело")
ax.set_title("Оптимальний транспортний план")
for i in range(result.plan.shape[0]):
    for j in range(result.plan.shape[1]):
        ax.text(j, i, f"{result.plan.iloc[i,j]:.0f}", ha="center", va="center")
fig.colorbar(im, ax=ax, label="Обсяг")
plt.show()


## 8. Сценарій: закриття маршруту S2 → D3


In [ ]:
closed = solve_transport_pulp(
    costs, supply, demand,
    forbidden_routes=[("S2", "D3")]
)
print("Baseline:", result.total_cost)
print("Route closure:", closed.total_cost)
display(closed.plan)


Контрольне значення: **570**.

Обговоріть: чому закриття лише одного маршруту перебудовує декілька потоків?


## 9. Сценарії вартості


In [ ]:
rows = [{"scenario":"baseline", "cost":result.total_cost}]

shock = costs.copy()
shock.loc["S2","D3"] += 4
rows.append({"scenario":"S2-D3 +4", "cost":solve_transport_pulp(shock, supply, demand).total_cost})

d4 = costs.copy()
d4["D4"] += 2
rows.append({"scenario":"D4 routes +2", "cost":solve_transport_pulp(d4, supply, demand).total_cost})

scenarios = pd.DataFrame(rows)
display(scenarios)

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(scenarios["scenario"], scenarios["cost"])
ax.set_ylabel("Загальні витрати")
ax.set_title("Чутливість оптимального плану")
plt.show()


## 10. Інтерпретація

Дайте відповіді:

1. Які маршрути реально використовує baseline?
2. Який маршрут виявився критичним у сценарії закриття?
3. Чи достатньо знати тільки \(Z^*\)?
4. Які припущення моделі можуть бути неприйнятними для реального дослідження?


## 11. Research transfer

Опишіть аналог структури `source → flow → destination → cost/risk` у власному дисертаційному дослідженні.
